In [92]:
# ================================================================
# STAGE 12 — ENSEMBLE FRAUD DETECTOR
# ================================================================

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import average_precision_score

print("=" * 70)
print("STAGE 12 — ENSEMBLE FRAUD DETECTOR")
print("=" * 70)

# ------------------------------------------------
# XGBoost score
# ------------------------------------------------

xgb_score = optimized_val_proba

# ------------------------------------------------
# Normalize anomaly score
# ------------------------------------------------

scaler = MinMaxScaler()

anomaly_score_norm = scaler.fit_transform(
    val_anomaly_score.reshape(-1, 1)
).ravel()

# ------------------------------------------------
# Test ensemble weights
# ------------------------------------------------

weights = np.arange(
    0.0,
    1.01,
    0.05
)

ensemble_results = []

for w in weights:

    ensemble_score = (
        w * xgb_score
        +
        (1 - w) * anomaly_score_norm
    )

    pr_auc = average_precision_score(
        y_val,
        ensemble_score
    )

    ensemble_results.append({
        "XGB_Weight": w,
        "Anomaly_Weight": 1 - w,
        "PR_AUC": pr_auc
    })

ensemble_df = pd.DataFrame(
    ensemble_results
)

best_ensemble = ensemble_df.loc[
    ensemble_df["PR_AUC"].idxmax()
]

print("\nBest ensemble:")
print(best_ensemble)

print("\nBaseline XGB PR-AUC:",
      average_precision_score(
          y_val,
          xgb_score
      ))

print(
    "Best ensemble PR-AUC:",
    best_ensemble["PR_AUC"]
)

ensemble_df.to_csv(
    "stage12_ensemble_results.csv",
    index=False
)

STAGE 12 — ENSEMBLE FRAUD DETECTOR

Best ensemble:
XGB_Weight        1.000000
Anomaly_Weight    0.000000
PR_AUC            0.814246
Name: 20, dtype: float64

Baseline XGB PR-AUC: 0.8142461161013738
Best ensemble PR-AUC: 0.8142461161013738


What you're looking for

If:

ensemble PR-AUC > XGB PR-AUC

then anomaly detection contributes complementary information.

If not, that's still an important finding.